## LG Aimers 9기 — 시간 검증 모델링 (Colab)

이 노트북은 현재 기준선인 LightGBM을 **과거 시즌 학습 → 다음 시즌 검증** 방식으로 실행하고 결과를 모은다. 기본 설정은 2019~2023으로 학습해 2024를 검증한다.

중요: 현재 LightGBM 기준선은 CPU 모델이다. Colab에서 GPU 런타임을 선택해도 자동으로 GPU를 쓰지 않는다. 대신 데이터/코드는 그대로 두고, 다음 실험에서 GPU를 직접 지원하는 CatBoost/XGBoost를 추가해 앙상블 후보로 비교한다. LightGBM은 1개 폴드가 비교적 짧아 CPU 기준선으로 유지한다.

### 0. 프로젝트 위치 확인

Colab에서는 저장소 전체를 `/content/Aimers_9th`에 두거나 Google Drive에 마운트한 뒤, 아래 후보 경로에 실제 위치를 추가한다. `data/train.csv`, `src/features.py`, `experiments`가 함께 있어야 한다.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import shutil
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/Aimers_9th"),
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/Aimers_9th"),
]
PROJECT_ROOT = next(
    (p.resolve() for p in candidates
     if (p / "data/train.csv").exists() and (p / "src/features.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트를 찾지 못했습니다. candidates에 Aimers_9th의 실제 경로를 추가하세요."
    )
print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /Users/hyejin/Aimers_9th


### 1. 의존성 설치 및 런타임 확인

루트 `requirements.txt`는 로컬/Colab 실험용이다. 제출 ZIP의 `baseline_submit/requirements.txt`와 목적이 다르다. Colab이 패키지 재시작을 안내하면 이 셀 실행 후 런타임을 한 번 재시작한다.

In [2]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")],
    check=True,
)
gpu_binary = shutil.which("nvidia-smi")
gpu = subprocess.run([gpu_binary], capture_output=True, text=True) if gpu_binary else None
print("GPU detected:", gpu is not None and gpu.returncode == 0)
if gpu is not None and gpu.returncode == 0:
    print(gpu.stdout.splitlines()[0])
print("Python:", sys.version.split()[0], "| CPU threads:", os.cpu_count())

GPU detected: False
Python: 3.11.9 | CPU threads: 8



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip


### 2. 시간 홀드아웃 실행

`VALID_SEASONS=[2024]`로 먼저 파이프라인을 확인한다. 안정성 검증 때 `[2022, 2023, 2024]`로 바꾼다. 모든 피처는 현재 행과 학습 폴드에서 정해진 상수만 사용하며 다른 validation/test 행의 통계를 사용하지 않는다.

In [3]:
VALID_SEASONS = [2024]  # 안정성 확인: [2022, 2023, 2024]
THREADS = max(1, min(8, os.cpu_count() or 2))
FORCE_RERUN = False

for valid_season in VALID_SEASONS:
    output_dir = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids"
    metrics_path = output_dir / "metrics.json"
    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[{valid_season}] 기존 결과 사용: {metrics_path}")
        continue

    cmd = [
        sys.executable, str(PROJECT_ROOT / "experiments/02_lgbm_holdout.py"),
        "--valid-season", str(valid_season),
        "--train-game-type", "R",
        "--exclude-player-ids",
        "--threads", str(THREADS),
        "--output-dir", str(output_dir),
    ]
    print("실행:", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

[2024] 기존 결과 사용: /Users/hyejin/Aimers_9th/outputs/lgbm_holdout_2024_train_R_no_ids/metrics.json


### 3. 폴드 결과 비교

공식 점수는 0 아래를 잘라내지만 실험에서는 음수도 원인을 보기 위해 `bss_raw`를 함께 기록한다. 모델 선택은 최신 한 폴드만 보지 말고 2022~2024의 Brier/BSS 방향과 평균 오차를 함께 본다.

In [4]:
import pandas as pd

rows = []
for valid_season in VALID_SEASONS:
    path = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids" / "metrics.json"
    with path.open(encoding="utf-8") as f:
        m = json.load(f)
    rows.append({
        "valid_season": valid_season,
        "train_rows": m["train_rows"],
        "brier": m["brier"],
        "bss_raw": m["bss_raw"],
        "roc_auc": m["roc_auc"],
        "target_rate": m["target_rate"],
        "prediction_mean": m["prediction_mean"],
        "mean_gap_pct_point": 100 * m["mean_gap"],
        "train_seconds": m["train_seconds"],
    })
summary = pd.DataFrame(rows).sort_values("valid_season")
summary

,valid_season,train_rows,brier,bss_raw,roc_auc,target_rate,prediction_mean,mean_gap_pct_point,train_seconds
0,2024,1090591,0.248196,644.686575,0.546954,0.486105,0.498748,1.264307,42.330325


### 4. 취약 구간 확인

전체 점수만 좋아도 `game_type=F`, 다음 시즌 신규 선수, 누적 투구 수가 작은 구간에서 무너질 수 있다. 이 표는 다음 실험의 개선 대상을 찾는 진단용이다. 하위 집단 점수를 단순 평균해 공식 점수로 해석하지 않는다.

In [5]:
last_season = max(VALID_SEASONS)
last_dir = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{last_season}_train_R_no_ids"
group_metrics = pd.read_csv(last_dir / "group_metrics.csv")
group_metrics[[
    "group", "value", "n", "target_rate", "prediction_mean",
    "brier", "bss_raw", "roc_auc"
]].head(30)

,group,value,n,target_rate,prediction_mean,brier,bss_raw,roc_auc
0,overall,all,253507,0.486105,0.498748,0.248196,644.686575,0.546954
1,game_type,F,30010,0.459280,0.480718,0.248058,114.220428,0.532130
2,game_type,R,223497,0.489707,0.501169,0.248215,671.899422,0.547633
3,pitcher_history,seen,195811,0.486249,0.498073,0.248225,634.761194,0.546173
4,pitcher_history,unseen,57696,0.485614,0.501040,0.248099,678.250164,0.549784
5,batter_history,seen,221587,0.488341,0.500254,0.248254,644.459134,0.546757
6,batter_history,unseen,31920,0.470583,0.488291,0.247798,536.341660,0.545117
7,cold_start,pitcher_n=0,81,0.407407,0.484546,0.246805,-2227.738121,0.505682
8,cold_start,batter_n=0,88,0.500000,0.492363,0.255357,-2142.617006,0.441632
9,asof_pitcher_n,0,81,0.407407,0.484546,0.246805,-2227.738121,0.505682


In [6]:
feature_importance = pd.read_csv(last_dir / "feature_importance.csv")
feature_importance.head(25)

,feature,importance_gain
0,smoothed_pitcher_success_rate,38712.907622
1,asof_pitcher_success_rate,38385.505262
2,pitcher_team_id,27373.524850
3,batter_team_id,25537.408534
4,asof_pitcher_reverse_rate,22998.196043
5,count_state,19761.202258
6,asof_pitcher_ball_rate,13122.917001
7,asof_pitcher_n,10984.091010
8,hand_matchup,10460.102108
9,season,10042.795396


### 5. 이 결과 다음에 할 일

1. 2022·2023·2024 세 폴드 결과를 고정 기준선으로 저장한다.
2. LightGBM의 학습 기간, 최근 시즌 가중치, `season` 포함 여부를 한 번에 하나씩 바꾼다.
3. Colab GPU를 직접 쓰는 CatBoost/XGBoost의 OOF 예측을 만든다.
4. 모델별 시간 OOF 예측만으로 calibration과 constrained blend 가중치를 학습한다.
5. 최종 선택 후 2019~2024 전체로 재학습하고 DACON 코드 제출 ZIP을 만든다.

현재 validation 정답 평균으로 예측을 shift하거나, test 전체 `pred.mean()`을 목표값에 맞추는 보정은 하지 않는다.